# Expiration Calendar and Calendar Spreads

Different futures expirations provide
the ability to match timing of risks with hedges,
but not all expirations have similar liquidity.
This notebook uses Databento to explore CME futures
liquidity by expiration for a few example markets.
After reading through this notebook, you should understand that
* each futures market has its own relationship between liquidity and expiration,
* data analysis and trading opportunities must adapt to the
  liquidity available in any particular market,
* liquidity across expirations is often reflected by liquidity in calendar spreads,
* cleared trading volume per expiration does not reflect spread trading,
* microstructure analysis on legs must account for spreads and implied trading.


In [34]:
import datetime
import sys
from collections.abc import Iterable
from functools import reduce
from itertools import cycle
from zoneinfo import ZoneInfo

import databento as db
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from pandas.tseries.holiday import USFederalHolidayCalendar
from pandas.tseries.offsets import CustomBusinessDay
from plotly.subplots import make_subplots

from finm37000 import (
    as_ct,
    disk_cache,
    filter_legs,
    get_cme_session_end,
    get_databento_api_key,
    get_official_stats,
    get_stats_by_date,
    temp_env,
)

# Example with US Federal Holidays
us_business = CustomBusinessDay(calendar=USFederalHolidayCalendar())

pio.templates.default = "plotly_dark"
px.defaults.color_discrete_sequence = px.colors.qualitative.Set3
color_palette = cycle(px.defaults.color_discrete_sequence)

sys.executable

'/Users/ericpatterson/Documents/github/finm37000-main/.venv/bin/python'

In [35]:
with temp_env(DATABENTO_API_KEY=get_databento_api_key()):
    client = db.Historical()

## Cleared Volume Across Time and Expiration

Let's start by looking at Crude and Gold.
Both contracts list every month for multiple years, yet
the trade volume for each expiration is quite different.

In [36]:
cme = "GLBX.MDP3"
products = (
    "CL",
    "GC",
)
# Pick some dates near, after, and then near the next expiration.
# Crude settles on the 3rd business day prior to the 25th of the month.
crude_near_roll = (
    datetime.date(2026, 4, 17),
    datetime.date(2026, 5, 4),
    datetime.date(2026, 5, 15),
)
# Gold settles on the 3rd to last business day of the month
gold_near_roll = (
    datetime.date(2026, 3, 23),
    datetime.date(2026, 4, 7),
    datetime.date(2026, 4, 21),
)
year = 2026
start_of_this_year = (datetime.date(2026, 1, 1) + us_business).date()
start_of_next_year = (datetime.date(2027, 1, 1) + us_business).date()
tz_chicago = ZoneInfo("America/Chicago")
now = datetime.datetime.now(tz=tz_chicago)

# Some examples below can be checked against the CME
# web site for the last five days. To keep this notebook
# working, you may need to update the expiration to check.
today = now.date()
yesterday = today - 2 * us_business
expiration_to_check_yesterday = "CLX6"

snippet_date = pd.Timestamp("2026-05-08", tz=tz_chicago)
# Note that the example_ids below are specific to
# this snippet. Modifying the data set will invalidate
# those examples.
snippet_start = snippet_date + pd.Timedelta(hours=13, minutes=28)
snippet_end = snippet_date + pd.Timedelta(hours=13, minutes=30)

In [37]:
cache_scope = "calendars"


@disk_cache(scope=cache_scope)
def cached_definitions(
    date: datetime.date,
    parent: str,
    dataset: str,
    client: db.Historical,
) -> pd.DataFrame:
    return client.timeseries.get_range(
        dataset=dataset,
        schema="definition",
        symbols=parent,
        stype_in="parent",
        start=date,
    ).to_df()


@disk_cache(scope=cache_scope)
def cached_stats_by_date(
    date: datetime.date,
    defs: pd.DataFrame,
    client: db.Historical,
) -> pd.DataFrame:
    return get_stats_by_date(
        client=client,
        date=date,
        defs=defs,
    )


futures_defs = {
    product: cached_definitions(
        date=start_of_this_year,
        parent=f"{product}.FUT",
        client=client,
        dataset=cme,
    )
    for product in products
}
futures_stats = {
    product: cached_stats_by_date(
        date=start_of_this_year,
        defs=defs,
        client=client,
    )
    for product, defs in futures_defs.items()
}
futures_legs = {product: filter_legs(defs) for product, defs in futures_defs.items()}

In [38]:
futures_groups = {
    product: tuple(
        df[df["expiration"].dt.date < start_of_next_year]
        .reset_index()["Symbol"]
        .unique(),
    )
    for product, df in futures_stats.items()
}

In [39]:
@disk_cache(scope=cache_scope)
def cached_stats(
    symbols: Iterable[str],
    start: datetime.date,
    end: pd.Timestamp,
    dataset: str,
    client: db.Historical,
) -> pd.DataFrame:
    return client.timeseries.get_range(
        dataset=dataset,
        schema="statistics",
        symbols=symbols,
        start=start,
        end=end,
    ).to_df()


symbols = reduce(lambda x, y: x + y, futures_groups.values())
inclusive_end = yesterday + us_business
raw_stats = cached_stats(
    dataset=cme,
    symbols=symbols,
    start=start_of_this_year,
    end=inclusive_end,
    client=client,
)

In [40]:
instrument_defs = pd.concat(
    [df.reset_index() for df in futures_legs.values()],
    ignore_index=True,
)
stats = get_official_stats(raw_stats, instrument_defs)

In [41]:
plot_df = stats.reset_index()
plot_groups = plot_df.groupby("Symbol")
colors = {group: next(color_palette) for group in plot_groups.groups}

x_col = "Trade date"
plot_cols = ["Cleared volume", "Open interest"]
subplot_titles = [f"{family} {col}" for family in futures_groups for col in plot_cols]

fig = make_subplots(
    rows=len(futures_groups),
    cols=len(plot_cols),
    shared_xaxes="all",
    subplot_titles=subplot_titles,
)

for i, (_, group_symbols) in enumerate(futures_groups.items()):
    for symbol in group_symbols:
        df = plot_groups.get_group(symbol)
        fig.add_trace(
            go.Scatter(
                x=df[x_col],
                y=df[plot_cols[0]],
                name=symbol,
                line=dict(color=colors[symbol]),
            ),
            row=i + 1,
            col=1,
        )
        fig.add_trace(
            go.Scatter(
                x=df[x_col],
                y=df[plot_cols[1]],
                name=symbol,
                line=dict(color=colors[symbol]),
                showlegend=False,
            ),
            row=i + 1,
            col=2,
        )

fig.update_layout(
    height=600,
    width=800,
    title_text=f"Cleared volume over time for {year} contracts",
)
fig.show()

* What do you notice about these graphs?
* What is similar and what is different?
* What about these products could explain these differences?

In [42]:
product = "CL"
dates = crude_near_roll
plot_df = stats.reset_index()
plot_df = plot_df[plot_df["Trade date"].isin(dates)]
plot_df = plot_df[plot_df["Symbol"].isin(futures_groups[product])]
plot_groups = plot_df.groupby("Trade date")
color = px.defaults.color_discrete_sequence[0]

x_col = "expiration"
plot_cols = ["Settlement price", "Cleared volume", "Open interest"]
short_names = ("Settle", "Volume", "OI")
subplot_titles = [f"{date}" for date in dates]

fig = make_subplots(
    rows=len(plot_cols),
    cols=len(dates),
    shared_xaxes="all",
    shared_yaxes=True,
    subplot_titles=subplot_titles,
    x_title="Expiration date",
)

for i, trade_date in enumerate(dates):
    df = plot_groups.get_group(trade_date)
    fig.add_trace(
        go.Scatter(
            x=df[x_col],
            y=df[plot_cols[0]],
            name=str(trade_date),
            line=dict(color=color),
            showlegend=False,
        ),
        row=1,
        col=i + 1,
    )
    fig.add_trace(
        go.Bar(
            x=df[x_col],
            y=df[plot_cols[1]],
            name=str(trade_date),
            marker=dict(color=color),
            showlegend=False,
        ),
        row=2,
        col=i + 1,
    )
    fig.add_trace(
        go.Bar(
            x=df[x_col],
            y=df[plot_cols[2]],
            name=str(trade_date),
            marker=dict(color=color),
            showlegend=False,
        ),
        row=3,
        col=i + 1,
    )

fig.update_layout(
    height=600,
    width=800,
    title_text=(
        f"Expiration volume and open interest comparison through {product} roll."
    ),
    yaxis_title="test",
)
for i, y_name in enumerate(short_names):
    fig.update_yaxes(title_text=y_name, row=i + 1, col=1)
fig.show()

In [43]:
product = "GC"
dates = gold_near_roll
plot_df = stats.reset_index()
plot_df = plot_df[plot_df["Trade date"].isin(dates)]
plot_df = plot_df[plot_df["Symbol"].isin(futures_groups[product])]
plot_groups = plot_df.groupby("Trade date")
color = px.defaults.color_discrete_sequence[0]

x_col = "expiration"
plot_cols = ["Settlement price", "Cleared volume", "Open interest"]
short_names = ("Settle", "Volume", "OI")
subplot_titles = [f"{date}" for date in dates]

fig = make_subplots(
    rows=len(plot_cols),
    cols=len(dates),
    shared_xaxes="all",
    shared_yaxes=True,
    subplot_titles=subplot_titles,
    x_title="Expiration date",
)

for i, trade_date in enumerate(dates):
    df = plot_groups.get_group(trade_date)
    fig.add_trace(
        go.Scatter(
            x=df[x_col],
            y=df[plot_cols[0]],
            name=str(trade_date),
            line=dict(color=color),
            showlegend=False,
        ),
        row=1,
        col=i + 1,
    )
    fig.add_trace(
        go.Bar(
            x=df[x_col],
            y=df[plot_cols[1]],
            name=str(trade_date),
            marker=dict(color=color),
            showlegend=False,
        ),
        row=2,
        col=i + 1,
    )
    fig.add_trace(
        go.Bar(
            x=df[x_col],
            y=df[plot_cols[2]],
            name=str(trade_date),
            marker=dict(color=color),
            showlegend=False,
        ),
        row=3,
        col=i + 1,
    )

fig.update_layout(
    height=600,
    width=800,
    title_text=(
        f"Expiration volume and open interest comparison through {product} roll."
    ),
    yaxis_title="test",
)
for i, y_name in enumerate(short_names):
    fig.update_yaxes(title_text=y_name, row=i + 1, col=1)
fig.show()

## Comparing Cleared Volume to the Trade Record

Total volume per day includes block trades and
other trades that do not show up in the electronic record as trades
for each expiration. Here is an example:

In [44]:
check_date = yesterday - us_business
session_end = get_cme_session_end(check_date)
session_start = session_end - datetime.timedelta(days=1)
print(
    f"{expiration_to_check_yesterday} in the session "
    f"from {session_start} to {session_end}."
)

CLX6 in the session from 2026-09-23 16:00:00-05:00 to 2026-09-24 16:00:00-05:00.


In [45]:
@disk_cache(scope=cache_scope)
def cached_trades(
    dataset: str,
    symbols: Iterable[str],
    start: pd.Timestamp,
    end: pd.Timestamp,
    client: db.Historical,
) -> pd.DataFrame:
    return client.timeseries.get_range(
        dataset=dataset,
        schema="trades",
        symbols=symbols,
        start=start,
        end=end,
    ).to_df()


leg_symbol = expiration_to_check_yesterday
leg_raw_trades = cached_trades(
    dataset=cme,
    symbols=[leg_symbol],
    start=session_start,
    end=session_end,
    client=client,
)

The quantity reported in the electronic trade log:

In [46]:
leg_raw_trades["size"].sum()

np.uint64(288490)

The quantity reported as cleared by the exchange:

In [47]:
stats[
    (stats.index.get_level_values(0) == check_date.to_pydatetime().date())
    & (stats.index.get_level_values(1) == leg_symbol)
]["Cleared volume"]

Trade date  Symbol
2026-09-24  CLX6      410015.0
Name: Cleared volume, dtype: float64

You can check that against the CME web site: https://www.cmegroup.com/markets/energy/crude-oil/light-sweet-crude.volume.html

The block trades are not enough to explain the difference.
Where is the missing volume?

## Listed Calendar Spread Trades

Listed calendar spreads are an efficient way
to trade one expiration against another.
The trade contributes to the cleared volume in its legs,
but there is no cleared spread volume.

In [48]:
product = "CL"
parent = f"{product}.FUT"
all_defs = cached_definitions(
    dataset=cme,
    parent=parent,
    date=yesterday.date(),
    client=client,
)

In [49]:
def_cols = [
    "symbol",
    "instrument_class",
    "secsubtype",
    "min_price_increment",
]
all_defs[def_cols]

,symbol,instrument_class,secsubtype,min_price_increment
ts_recv,,,,
2026-09-25 00:00:00+00:00,CLM7-WSM7,S,IS,0.01
2026-09-25 00:00:00+00:00,CLM7-WSM7,S,IS,0.01
2026-09-25 00:00:00+00:00,CLK7-WSK7,S,IS,0.01
2026-09-25 00:00:00+00:00,CLK7-WSK7,S,IS,0.01
2026-09-25 00:00:00+00:00,CL:SA 03M J9,S,SA,0.01
...,...,...,...,...
2026-09-25 16:27:01.396784207+00:00,UD:CL: GN 2653080,S,GN,0.01
2026-09-25 16:27:01.396784207+00:00,UD:CL: GN 2653080,S,GN,0.01
2026-09-25 17:22:19.107699490+00:00,UD:CL: GN 2655075,S,GN,0.01


Crude is a great example of a market with heavy spread trading,
but there is one feature that it does not
demonstrate here. The minimum price increment is `0.01`
for Crude futures and spreads, but in some markets,
the spreads may trade at smaller increments!

In [50]:
front_leg_defs = all_defs[all_defs["symbol"].str.contains(leg_symbol[2:])]
front_spreads = front_leg_defs["symbol"].unique()
front_spreads

<ArrowStringArray>
[         'CLX6-WSX6',          'CLX6-BZX6',          'CLV6-CLX6',
     'CL:C1 RB-CL X6',     'CL:C1 HO-CL X6',          'CLX6-CLZ6',
               'CLX6',     'CL:BF X6-Z6-F7',          'CLX6-CLF7',
         'CLX6-WTTX6',         'CLX6-HTTX6',       'CL:SA 04M X6',
          'CLX6-CLG7',          'CLX6-BZZ6',          'CLV6-BZX6',
          'CLX6-BZF7',          'CLX6-CLH9',          'CLX6-CLV8',
          'CLX6-CLJ8',          'CLX6-CLF9',       'CL:SA 02M X6',
         'CLX6-MCLX6',       'CL:SA 12M X6',     'CL:BF X6-G7-K7',
 'WS:XS 02M CL-BZ X6',          'CLX6-CLH7',          'CLX6-CLJ7',
          'CLX6-CLN7',          'CLX6-CLH0',          'CLX6-CLU9',
          'CLX6-CLZ7',          'CLX6-CLM8',          'CLX6-CLU8',
          'CLX6-CLM9',          'CLX6-CLF8',          'CLX6-CLQ8',
          'CLX6-CLK9',          'CLX6-CLK7',          'CLX6-CLH8',
          'CLX6-CLQ7',          'CLX6-CLV7',          'CLX6-CLM7',
          'CLX6-CLQ9',          'CLX6-CLZ8'

In [51]:
@disk_cache(scope=cache_scope)
def cached_tbbo(
    dataset: str,
    parent: str,
    start: pd.Timestamp,
    end: pd.Timestamp,
    client: db.Historical,
) -> pd.DataFrame:
    return client.timeseries.get_range(
        dataset=dataset,
        schema="tbbo",
        symbols=parent,
        stype_in="parent",
        start=start,
        end=end,
    ).to_df()


trade_snippet = cached_tbbo(
    dataset=cme,
    parent=parent,
    start=snippet_start,
    end=snippet_end,
    client=client,
)

In [52]:
trade_snippet["local_time"] = as_ct(trade_snippet["ts_event"])
trade_local = trade_snippet.set_index(["local_time", "sequence"])
index_df = trade_local.index.to_frame()
trade_local["time_sequence_id"] = index_df.ne(index_df.shift(1)).any(axis=1).cumsum()
trade_local["group_size"] = trade_local.groupby(["time_sequence_id"]).transform("size")

In [53]:
trade_volume_by_symbol = (
    trade_local.groupby("symbol")[["size"]].sum().sort_values("size", ascending=False)
)
fig = px.bar(trade_volume_by_symbol, x=trade_volume_by_symbol.index, y="size")
fig.update_layout(
    title=dict(
        text=f"{product} Trade Volume by Symbol",
        subtitle=dict(
            text=f"Between {snippet_start} and {snippet_end}",
        ),
    )
)

fig.show()

In [54]:
front_trades = trade_local[trade_local["symbol"].isin(front_spreads)]
front_trade_volume_by_symbol = (
    front_trades.groupby("symbol")[["size"]].sum().sort_values("size", ascending=False)
)
fig = px.bar(
    front_trade_volume_by_symbol, x=front_trade_volume_by_symbol.index, y="size"
)
fig.update_layout(
    title=dict(
        text=f"{leg_symbol} Trade Volume by Symbol",
        subtitle=dict(
            text=f"Between {snippet_start} and {snippet_end}",
        ),
    )
)
fig.show()

## Implied Trades

Implied trades are trades in a leg or spread that are
initiated by a trade in a different leg or spread.

Side:
* `A`: Ask-initiated trade, i.e., sell, typically at the bid price.
* `B`: Bid-initiated trade, i.e., buy, typically at the ask price.
* `N`: None; in these examples, that means implied fills.

CME
https://cmegroupclientsite.atlassian.net/wiki/spaces/EPICSANDBOX/pages/457422235/MDP+3.0+-+Implied+Book

Databento
https://databento.com/docs/standards-and-conventions/common-fields-enums-types#side?historical=python&live=python&reference=python

Let's look at some examples during this 2-minute snippet.

In [55]:
cols = [
    "time_sequence_id",
    "group_size",
    "symbol",
    "side",
    "size",
    "price",
    "bid_ct_00",
    "bid_sz_00",
    "bid_px_00",
    "ask_px_00",
    "ask_sz_00",
    "ask_ct_00",
]
example_ids = [3636, 782, 3539]
by_timestamp = trade_local.copy()[cols]

### Implied Out Calendar Example

Buying `X-Y` at price `p` typically means buying `X` and
selling `Y`, and the price `p` is the difference of their prices,
`p=X-Y` (conflating the contract symbol and its price).
There are markets that follow opposite conventions (e.g.,
equity index futures), but this
convention is simple to follow and applies in the crude market.

In this example, an aggressive sell order (`side==A`)
for 14 `CLZ6` futures
gets filled at two price levels:
5 for \$79.45 and 9 for \$79.44.
All six rows share a timestamp and a sequence number, so they
are one matching event.
The first 5 match the displayed bid of 5 at \$79.45 (`bid_sz_00`).
The second level fill of 9 at \$79.44 includes 5 implied trades:

* Fill 3 of the `CLM7` bid (\$74.38) and the `CLZ6-CLM7` bid (\$5.06)
  implying a bid of \$74.38 + \$5.06=\$79.44.
* Fill 2 of the `CLM6` bid (\$95.56) and the `CLM6-CLZ6` offer (\$16.12)
  implying a bid of \$95.56 - \$16.12 = \$79.44.

By reading the corresponding
implied fills, we see that the 9 `CLZ6` traded at \$79.44
consist of 5 implied fills and the remaining 4 resting
must have been real orders.

The implied trades here are "implied out" meaning they use
a spread and a leg to imply the other leg. The next example
includes an "implied in" trade where two legs imply into the
spread market.

In [56]:
by_timestamp.loc[by_timestamp["time_sequence_id"] == example_ids[0], cols]

time_sequence_id  group_size  \
local_time                          sequence                                  
2026-05-08 13:28:34.501490229-05:00 997840833              3636           6   
                                    997840833              3636           6   
                                    997840833              3636           6   
                                    997840833              3636           6   
                                    997840833              3636           6   
                                    997840833              3636           6   

                                                  symbol side  size  price  \
local_time                          sequence                                 
2026-05-08 13:28:34.501490229-05:00 997840833       CLZ6    A     5  79.45   
                                    997840833       CLZ6    A     9  79.44   
                                    997840833       CLM6    N     2  95.56   
                                    997840833  CLM6-CLZ6    N     2  16.12   
                                    997840833       CLM7    N     3  74.38   
                                    997840833  CLZ6-CLM7    N     3   5.06   

                                               bid_ct_00  bid_sz_00  \
local_time                          sequence                          
2026-05-08 13:28:34.501490229-05:00 997840833          5          5   
                                    997840833          5          5   
                                    997840833          2          2   
                                    997840833          3          5   
                                    997840833          3          3   
                                    997840833          2          3   

                                               bid_px_00  ask_px_00  \
local_time                          sequence                          
2026-05-08 13:28:34.501490229-05:00 997840833      79.45      79.47   
                                    997840833      79.45      79.47   
                                    997840833      95.56      95.57   
                                    997840833      16.05      16.12   
                                    997840833      74.38      74.40   
                                    997840833       5.06       5.08   

                                               ask_sz_00  ask_ct_00  
local_time                          sequence                         
2026-05-08 13:28:34.501490229-05:00 997840833          3          3  
                                    997840833          3          3  
                                    997840833          2          1  
                                    997840833          2          2  
                                    997840833          1          1  
                                    997840833         12         10

### Legs, Calendar, and Butterfly Example

In this example, the `CLM6-CLN6` trades 20 at the ask \$3.62
on a buy aggressor. That includes fills on

* 16 real `CLM6-CLN6` offers for \$3.62
* The legs of the spread implied IN for the spread \$95.41 - \$91.79 = \$3.62.
    * Leg `CLM6`, aggressor lifted one offer for \$95.41
    * Leg `CLN6`, aggressor hit one bid for \$91.79
* 3 spread and butterfly offers implied the -\$0.02 + \$3.64 = \$3.62 in the `M6-N6` spread.
    * Butterfly `CL:BF M6-N6-Q6`, aggressor lifted 3 offers for -\$0.02.
        * Butterfly is long one `M6-N6`, short one `N6-Q6` (`BF = S1 - S2`)
        * Buy `M6-N6` by buying butterfly and `N6-Q6` (`S1 = BF + S2`)
    * Spread `CLN6-CLQ6`, aggressor lifted 3 offers for \$3.64.


In [57]:
by_timestamp.loc[by_timestamp["time_sequence_id"] == example_ids[1], cols]

time_sequence_id  group_size  \
local_time                          sequence                                  
2026-05-08 13:28:03.660860499-05:00 997320565               782           5   
                                    997320565               782           5   
                                    997320565               782           5   
                                    997320565               782           5   
                                    997320565               782           5   

                                                       symbol side  size  \
local_time                          sequence                               
2026-05-08 13:28:03.660860499-05:00 997320565       CLM6-CLN6    B    20   
                                    997320565            CLM6    N     1   
                                    997320565            CLN6    N     1   
                                    997320565       CLN6-CLQ6    N     3   
                                    997320565  CL:BF M6-N6-Q6    N     3   

                                               price  bid_ct_00  bid_sz_00  \
local_time                          sequence                                 
2026-05-08 13:28:03.660860499-05:00 997320565   3.62          4          8   
                                    997320565  95.41          1          1   
                                    997320565  91.79          1          1   
                                    997320565   3.64          4          5   
                                    997320565  -0.02          3          3   

                                               bid_px_00  ask_px_00  \
local_time                          sequence                          
2026-05-08 13:28:03.660860499-05:00 997320565       3.61       3.62   
                                    997320565      95.40      95.41   
                                    997320565      91.79      91.81   
                                    997320565       3.63       3.64   
                                    997320565      -0.05      -0.02   

                                               ask_sz_00  ask_ct_00  
local_time                          sequence                         
2026-05-08 13:28:03.660860499-05:00 997320565         16          9  
                                    997320565          1          1  
                                    997320565          2          2  
                                    997320565          4          3  
                                    997320565          5          3

Note how the aggressor to buy 20 `CLM6-CLN6`
corresponds to each participant's fills in the legs:

| Participant | `CLM6` | `CLN6` | `CLQ6` |
|-------------|--------|--------|--------|
| Aggressor (buys 20 `M6-N6`) | +20 | -20 | |
| 16 real `M6-N6` offers | -16 | +16 | |
| 1 `M6` offer (implied in) | -1 | | |
| 1 `N6` bid (implied in) | | +1 | |
| 3 `CL:BF M6-N6-Q6` offers | -3 | +6 | -3 |
| 3 `N6-Q6` offers | | -3 | +3 |
| Volume | 20 | 23 | 3 |

In [58]:
# Using Pandas to render a table here because
# VS Code stopped rendering the markdown table
# the night before class!
leg_fills = pd.DataFrame(
    {
        "CLM6": [20, -16, -1, 0, -3, 0],
        "CLN6": [-20, 16, 0, 1, 6, -3],
        "CLQ6": [0, 0, 0, 0, -3, 3],
    },
    index=pd.Index(
        [
            "Aggressor (buys 20 M6-N6)",
            "16 real M6-N6 offers",
            "1 M6 offer (implied in)",
            "1 N6 bid (implied in)",
            "3 CL:BF M6-N6-Q6 offers",
            "3 N6-Q6 offers",
        ],
        name="Participant",
    ),
)
net = leg_fills.sum()
volume = leg_fills.clip(lower=0).sum()
leg_fills.loc["Net"] = net
leg_fills.loc["Volume"] = volume
leg_fills

,CLM6,CLN6,CLQ6
Participant,,,
Aggressor (buys 20 M6-N6),20,-20,0
16 real M6-N6 offers,-16,16,0
1 M6 offer (implied in),-1,0,0
1 N6 bid (implied in),0,1,0
3 CL:BF M6-N6-Q6 offers,-3,6,-3
3 N6-Q6 offers,0,-3,3
Net,0,0,0
Volume,20,23,3


The columns show both sides of every fill, so the
fills cancel per column. The volume in each leg is the
sum of one side.
Note that the butterfly and `N6-Q6` offers partly trade
with each other rather than with the aggressor, so breaking
down the legs of each fill separately would double count
`CLN6` and `CLQ6`.

If you look through trade data for `symbol == "CLM6"`,
you would only account for one of those futures.

Alternatively, you could attempt to track all
fills across all spreads involving the leg,
but then you need to be sure not to double count fills that are implied.

Feel free to look at other times,
group sizes, or products for as many examples as you need until
this makes sense. For example, can you work out how
each leg filled in this example:

In [59]:
by_timestamp.loc[by_timestamp["time_sequence_id"] == example_ids[2], cols]

time_sequence_id  group_size  \
local_time                          sequence                                  
2026-05-08 13:28:33.333045835-05:00 997824891              3539           9   
                                    997824891              3539           9   
                                    997824891              3539           9   
                                    997824891              3539           9   
                                    997824891              3539           9   
                                    997824891              3539           9   
                                    997824891              3539           9   
                                    997824891              3539           9   
                                    997824891              3539           9   

                                                       symbol side  size  \
local_time                          sequence                               
2026-05-08 13:28:33.333045835-05:00 997824891            CLQ6    B     5   
                                    997824891            CLN6    N     5   
                                    997824891       CLN6-CLQ6    N     5   
                                    997824891            CLQ6    B     4   
                                    997824891            CLM6    N     2   
                                    997824891       CLM6-CLQ6    N     2   
                                    997824891            CLN6    N     1   
                                    997824891            CLU6    N     1   
                                    997824891  CL:BF N6-Q6-U6    N     1   

                                               price  bid_ct_00  bid_sz_00  \
local_time                          sequence                                 
2026-05-08 13:28:33.333045835-05:00 997824891  88.26          3          3   
                                    997824891  91.90          7         10   
                                    997824891   3.64          1          5   
                                    997824891  88.27          3          3   
                                    997824891  95.54          5         10   
                                    997824891   7.27          2          2   
                                    997824891  91.91          7         10   
                                    997824891  85.38          1          1   
                                    997824891   0.75          4          9   

                                               bid_px_00  ask_px_00  \
local_time                          sequence                          
2026-05-08 13:28:33.333045835-05:00 997824891      88.24      88.28   
                                    997824891      91.89      91.90   
                                    997824891       3.64       3.65   
                                    997824891      88.24      88.28   
                                    997824891      95.53      95.54   
                                    997824891       7.27       7.30   
                                    997824891      91.89      91.90   
                                    997824891      85.34      85.38   
                                    997824891       0.75       0.76   

                                               ask_sz_00  ask_ct_00  
local_time                          sequence                         
2026-05-08 13:28:33.333045835-05:00 997824891          1          1  
                                    997824891          5          3  
                                    997824891          6          5  
                                    997824891          1          1  
                                    997824891          5          5  
                                    997824891          5          5  
                                    997824891          5          3  
                                    997824891          1          1  

### Determining fills

Some microstructure modeling needs to
consider the mechanics of which orders, real or implied, are filled. Here
are some resources you can consult.

Who gets filled? Matching algorithms vary by market. Some examples:
https://databento.com/blog/cme-matching-algorithms-explained
https://cmegroupclientsite.atlassian.net/wiki/spaces/EPICSANDBOX/pages/457087723/Complex+Match+Example

Which spreads are prioritized for implied trading?
https://cmegroupclientsite.atlassian.net/wiki/spaces/EPICSANDBOX/pages/457096650/Futures+Implied+Order+Matching+Priority

The basic ideas are
* real orders typically have priority over implied orders,
* matching in any particular market will reflect one or
  more of the following mechanisms
   * FIFO: Earlier get priority, rewards speed
   * Pro Rata: Fills are proportional to order size, rewards size
   * Top Order: First order at a price level gets priority,
     rewards aggressive price improvement
   * Lead market maker (LMM): certain market participants
     have priority to encourage participation.